# Análisis exploratorio: BAF Base

**Objetivo.** Conocer los datos antes de modelar y fijar las decisiones de
preprocesamiento con evidencia. Las conclusiones se trasladan a
`docs/datos.md` y a la memoria del TFM.

**Contenido**
1. Carga de los datos
2. Prevalencia de fraude por mes y por partición
3. Valores ausentes: ¿la ausencia se relaciona con el fraude?
4. Deriva entre meses
5. Atributo protegido (`customer_age >= 50`)

## 1. Carga de los datos

Se carga `Base.csv` con las rutas de `configs/local.yaml`. `resolve_path`
convierte la ruta en absoluta desde la raíz del repositorio, para que funcione
aunque el notebook se ejecute desde `notebooks/`.

In [2]:
import pandas as pd

from src.config import load_config, resolve_path
from src.data import load_data

config = load_config(resolve_path("configs/local.yaml"))
df = load_data(config)

df.shape


(1000000, 32)

**Resultado.** 1.000.000 de filas y 32 columnas, como indica la documentación
del BAF. La verificación de tipos, rangos y ausentes codificados como negativos
está en `docs/datos.md`.

## 2. Prevalencia de fraude por mes y por partición

Partición fija del proyecto: entrenamiento = meses 0–4, validación = mes 5,
test = meses 6–7.

**Qué se mira.** El número de solicitudes y la tasa de fraude en cada mes y en
cada partición.

**Por qué importa.** El umbral se fija en validación (5 % de FPR) y se aplica
tal cual al test. Si la distribución cambia entre el mes 5 y los meses 6–7, ese
umbral no dará exactamente un 5 % de FPR en test.

In [ ]:
import numpy as np

def particionar_df(x):
    if x <= 4:
        return "train"
    elif x == 5:
        return "eval"
    elif 5 < x <=7:
        return "test"
    else:
        return 0

df["split"] = df["month"].apply(particionar_df)


train = df[df["split"].str.contains("train")].copy().reset_index(drop=True)
eval = df[df["split"].str.contains("eval")].copy().reset_index(drop=True)
test = df[df["split"].str.contains("test")].copy().reset_index(drop=True)

print(len(train))



675666


## 3. Valores ausentes: ¿la ausencia se relaciona con el fraude?

**Qué se mira.**
- En las columnas con -1 como ausente, la tasa de fraude con y sin el dato.
- En `intended_balcon_amount`, la tasa de fraude por tramos de los valores
  negativos, para decidir si son ausentes (decisión pendiente en
  `docs/datos.md`).

**Por qué importa.** Si la ausencia predice el fraude, el indicador de
ausencia que usarán las redes aporta información.

## 4. Deriva entre meses

**Qué se mira.** Si la distribución de las variables cambia de un mes a otro,
sobre todo entre entrenamiento (0–4) y test (6–7).

**Por qué importa.** El modelo se entrena con el pasado y se evalúa con el
futuro. Una deriva fuerte explicaría caídas de rendimiento en test.

## 5. Atributo protegido: edad

Grupos: `customer_age >= 50` frente a `< 50`, el corte del código oficial del
BAF (ver `docs/datos.md`).

**Qué se mira.** El tamaño de cada grupo y su tasa de fraude, en total y por
partición.

**Por qué importa.** La métrica de equidad es el cociente de FPR entre los
dos grupos. Un grupo pequeño en test da un FPR con mucha varianza.